# MoshiRAG の学習データ作成をたどる

1 つの QA から parquet の 1 行ができるまでを、工程ごとに**実際に実行して**中身を見る。
GPU・TTS・MFA が要る工程（s04/s05/s07/s09）は、それぞれ専用の環境へ subprocess で渡す。

前提（g21 / kiso）:
- s02（対話生成）：ローカル vLLM が `http://localhost:8000/v1` で起動していること（自分の tmux で）
- s04（音声化）：`~/moshirag_tts/src/zoom1-tts`（専用 venv・FireRedTTS、モデルは HF キャッシュ）
- s05（アライン）：conda 環境 `~/miniforge3/envs/mfa` と日本語モデル（`~/moshirag_mfa/mfa_root`）
- s07/s09（音声トークン・参照ベクトル）：`~/moshirag_tok/moshirag-jp/.venv`（torch+kame）と空き GPU

In [1]:
# 準備：リポジトリのルートを探し、moshirag_data/ と tools/ を import できるようにする
import json, math, re, shutil, sys, warnings, wave
from argparse import Namespace
from pathlib import Path
from unittest import mock

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore", message="IProgress not found")   # tqdm の表示に関する警告

REPO = Path.cwd()
while not (REPO / "moshirag_data").is_dir():
    if REPO.parent == REPO:
        raise RuntimeError("リポジトリのルートが見つからない。リポジトリの中でノートブックを開く")
    REPO = REPO.parent
sys.path[:0] = [str(REPO / "moshirag_data"), str(REPO)]

import s01c_wiki_qa as s01c
import s02_dialogue as s02
import s02b_rebuild_references as s02b
import s03_tts_input as s03
import s05_align as s05
import s06_ret as s06
import s10_pack as s10
import tools.tokenize_text as tok_text   # s08 の実装本体（s08_tok_text.py はこれを呼ぶだけ）

WORK = REPO / "data" / "_walkthrough"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)

STEM = "wiki_qa_000000"     # s02 が付ける会話の ID（入力ファイル名 + 通し番号 6 桁）
FRAME_RATE = 12.5           # Mimi のフレームレート。s06〜s10 はすべてこの時間軸で揃える

pd.set_option("display.max_colwidth", 90)
_jp = [f for f in ("Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic")
       if any(x.name == f for x in font_manager.fontManager.ttflist)]
if _jp:
    plt.rcParams["font.family"] = _jp[0]


def ls(d: Path):
    # フォルダの中身を一覧にする（サブフォルダは 1 段だけ）
    rows = []
    for p in sorted(Path(d).rglob("*")):
        if p.is_file() or p.is_symlink():
            rows.append({"ファイル": str(p.relative_to(d)), "バイト": p.lstat().st_size})
    return pd.DataFrame(rows)


print("リポジトリ:", REPO)
print("作業場所  :", WORK)

リポジトリ: /home/jsato/moshirag-jp
作業場所  : /home/jsato/moshirag-jp/data/_walkthrough


In [23]:
# 実処理（GPU・TTS・MFA）を、それぞれ専用の環境へ subprocess で渡すための設定
import os, subprocess

GPU = os.environ.get("WALKTHROUGH_GPU", "0")   # 使う GPU。空きのある方を nvidia-smi で確かめて選ぶ

HOME = Path.home()
TTS_PKG   = HOME / "moshirag_tts" / "src" / "zoom1-tts"                       # TTS パッケージ（専用 venv つき）
TTS_PY    = TTS_PKG / ".venv" / "bin" / "python"
TTS_SYNTH = HOME / "moshirag_tts" / "synth_worker.py"                         # s04 の実装（モデルを 1 回だけ読む）
MFA_ENV   = HOME / "miniforge3" / "envs" / "mfa"                              # MFA の conda 環境
MFA_ROOT  = HOME / "moshirag_mfa" / "mfa_root"                                # 日本語モデルと設定の置き場
BEAM_YAML = HOME / "mfa_beam.yaml"                                            # beam 100 / retry_beam 400
TOK_PY    = HOME / "moshirag_tok" / "moshirag-jp" / ".venv" / "bin" / "python"  # torch+kame（s07/s09）

for p in (TTS_PY, TTS_SYNTH, MFA_ENV / "bin" / "mfa", MFA_ROOT, BEAM_YAML, TOK_PY):
    assert p.exists(), f"見つからない: {p}（README の前提の環境がこのマシンにあるか確認）"


def run_step(cmd, env=None, cwd=None, title=""):
    # 別環境のコマンドを実行し、出力を見せる。失敗したら末尾を出して止める
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    e = {**os.environ, **(env or {})}
    r = subprocess.run(cmd, cwd=(str(cwd) if cwd else None), env=e,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    tail = r.stdout[-4000:]
    if len(r.stdout) > 4000:
        tail = "…（省略）…\n" + tail
    print(tail)
    if r.returncode != 0:
        raise RuntimeError(f"{title or cmd[0]} が終了コード {r.returncode} で失敗")
    return r


print("GPU:", GPU, "/ 実環境はすべて見つかった")

GPU: 0 / 実環境はすべて見つかった


---
## s01　Wikipedia の記事から QA を作る

会話の種になる問いと答えの組を、Wikipedia（ja_wiki）の記事から LLM で作る。
どんな話題にも答えられるよう、記事を事前に絞り込むことはせず、すべての記事を LLM に渡す。

### 記事を読み込む

ja_wiki の本体は ABCI にしかないので、`moshirag_data/tmp/` に置いたサンプル記事（ja_wiki と同じ形）を使う。
s01c の入力の形（wiki_id / title / url / lead）にする。lead は本文の冒頭 1,000 文字。

In [2]:
S01 = WORK / "s01"
S01.mkdir()
LEAD_CHARS = 1000   # LLM に渡す本文の冒頭の長さ

samples = [json.loads(p.read_text(encoding="utf-8")) for p in sorted((REPO / "moshirag_data" / "tmp").glob("*.txt"))]
candidates = [{"wiki_id": d["meta"]["id"], "title": d["meta"]["title"], "url": d["meta"]["url"],
               "lead": d["text"][:LEAD_CHARS], "n_chars": len(d["text"])} for d in samples]
CAND = S01 / "wiki_articles.jsonl"
CAND.write_text("".join(json.dumps(c, ensure_ascii=False) + "\n" for c in candidates), encoding="utf-8")
pd.DataFrame([{"id": c["wiki_id"], "タイトル": c["title"], "本文の文字数": c["n_chars"],
               "本文の冒頭": c["lead"][:60]} for c in candidates])

,id,タイトル,本文の文字数,本文の冒頭
0,564184,川平有料道路,605,川平有料道路（かわひらゆうりょうどうろ）は、長崎県長崎市を通る一般有料道路である。長崎県道路公社が管理している。\n\n概要
1,1114111,風間完,528,風間 完（かざま かん、1919年1月19日 - 2003年12月27日）は、日本の挿絵画家である。美人画、風景画に加え
2,2132952,加良まゆみ,1031,加良 まゆみ（かりょう まゆみ、1962年8月21日 -）は、日本の女性声優。広島県広島市出身。TABプロダクション所属


### 記事から LLM で QA を作る

読み込んだ記事を、本番と同じ `s01c.main` で実際の LLM（`:8000` の vLLM）に渡し、問いと答えを作らせる。
題材にならない記事には LLM が「スキップ」と返し、出力に残らない。

In [3]:
# s01c と s02 が呼ぶ LLM（ローカル vLLM）。:8000 が実際に公開しているモデル名に合わせる
# （名前が違うと vLLM が 404 を返すため）
import urllib.request

LLM_BASE_URL = "http://localhost:8000/v1"
served = json.loads(urllib.request.urlopen(LLM_BASE_URL.rstrip("/") + "/models").read())["data"]
LLM_MODEL = served[0]["id"]
print("vLLM のモデル:", LLM_MODEL)

vLLM のモデル: google/gemma-4-E4B-it


In [4]:
# 本番（jobs/qa_gen/wikiqa_main.pbs）と同じ引数で s01c.main を流す（件数が少ないので workers だけ 1）
QA_ALL = S01 / "wiki_qa_all.jsonl"
argv = ["s01c_wiki_qa.py", "--input_file", str(CAND), "--output_file", str(QA_ALL),
        "--model", LLM_MODEL, "--llm_base_url", LLM_BASE_URL,
        "--role", "default", "--effort", "low", "--workers", "1"]
with mock.patch.object(sys, "argv", argv):
    s01c.main()

  判定プロンプト: default / 候補 3 件 / シャード 0/1 担当 3 件



  採用 3 / LLM スキップ 0 / 既存 0 / 失敗 0
  採用率 100.0%（既存を除く 3 件に対して）


In [5]:
# どの記事が採用され、どんな問いと答えになったか
made = {r["title"]: r for r in map(json.loads, QA_ALL.read_text(encoding="utf-8").splitlines())}
pd.DataFrame([{"記事": c["title"], "結果": "採用" if c["title"] in made else "スキップ",
               "問い": made.get(c["title"], {}).get("question", ""),
               "答え": made.get(c["title"], {}).get("answer", "")} for c in candidates])

,記事,結果,問い,答え
0,川平有料道路,採用,川平有料道路はどこを結んでいるの？,長崎バイパス川平ICと時津ランプ
1,風間完,採用,朝の連続テレビ小説「おはなはん」のタイトル画を描いたのは誰ですか？,風間完
2,加良まゆみ,採用,加良まゆみさんは、以前はどこの事務所に所属していましたか？,エーエス企画


### s02 に渡す QA を 1 件選ぶ

In [7]:
# 採用された QA から 1 件（サンプル記事の順で最初のもの）を s02 の入力にする
assert made, "LLM がすべての記事をスキップした。もう一度実行するか、tmp/ のサンプル記事を変える"
seed_qa = next(made[c["title"]] for c in candidates if c["title"] in made)
(S01 / "wiki_qa.jsonl").write_text(json.dumps(seed_qa, ensure_ascii=False) + "\n", encoding="utf-8")
print((S01 / "wiki_qa.jsonl").read_text(encoding="utf-8"))

{"question": "川平有料道路はどこを結んでいるの？", "answer": "長崎バイパス川平ICと時津ランプ", "source": "ja_wiki", "category": "wiki_generated", "wiki_id": "564184", "title": "川平有料道路", "url": "https://ja.wikipedia.org/wiki/%E5%B7%9D%E5%B9%B3%E6%9C%89%E6%96%99%E9%81%93%E8%B7%AF", "index": 2}



---
## s02　QA からテキスト対話データを作成する

LLM を役割ごとに呼び、話題 → 参照 3 本 → 素の会話 → 判定 → lead の順に作る。

本番と同じ引数で `s02.main` を実行。

In [8]:
S02 = WORK / "s02"
args02 = s02.build_parser().parse_args([
    "--input_file", str(S01 / "wiki_qa.jsonl"), "--output_dir", str(S02),
    "--num_samples", "1", "--seed", "1", "--model", LLM_MODEL, "--llm_base_url", LLM_BASE_URL,
    "--min_turns", "3", "--max_turns", "5", "--workers", "1",
    "--effort_human", "high", "--effort_assistant", "low", "--effort_chunk", "low",
    "--effort_judge", "low", "--effort_lead", "low",
])
s02.main(args02)

  シャード 0/1: 担当 1 件 / 全 1 件


[ok] wiki_qa_000000 ok turns=4 lines=12

Done: 1 ok / 0 fail -> /home/jsato/moshirag-jp/data/_walkthrough/s02


出力結果の中身を見る。

In [34]:
conv = json.loads((S02 / f"{STEM}.json").read_text(encoding="utf-8"))
for k in ("seed_question", "seed_answer", "topic", "persona", "judge_raw", "chunk_used",
          "n_user_turns", "ended_by_eoc", "code_commit", "code_dirty"):
    print(f"{k:14s}: {conv[k]!r}")
print("\n参照（chunks）:")
for i, c in enumerate(conv["chunks"], 1):
    print(f"  [{i}] {c}")
print("\n素の会話（plain）:")
for t in conv["plain"]:
    print(f"  {t['speaker']}: {t['text']}")

seed_question : '川平有料道路はどこを結んでいる道路ですか？'
seed_answer   : '長崎バイパス川平ICと時津ランプ'
topic         : '特定の有料道路の路線情報'
persona       : '40代以上 / ぶっきらぼう / 詳しい / なんとなく'
judge_raw     : '1: なし\n2: なし\n3: 3\n4: 2'
chunk_used    : {'1': [], '2': [], '3': [3], '4': [2]}
n_user_turns  : 4
ended_by_eoc  : True
code_commit   : 'ce7170f'
code_dirty    : True

参照（chunks）:
  [1] 川平有料道路は、長崎バイパス川平ICと時津ランプを結ぶ道路であり、長崎県内の交通アクセスを向上させている。
  [2] 川平有料道路の路線情報は限定的で、具体的にどの市町村間を結ぶのかは判明していない。
  [3] 川平有料道路は、長崎県北部の地域間を結び、地元の物流や観光客の移動を円滑化する役割を担っている。

素の会話（plain）:
  moshi: やあ。
  人間: あの道の、一番長い区間ってどれなんだろう
  moshi: 申し訳ないけど、川平有料道路の一番長い区間については知らないんだ。
  人間: 他に何か分かってる区間とかないの？
  moshi: 北部の地域間を結んでるってことは言えるけどね。
  人間: 具体的にどのエリア間を走ってるのか教えてくれる？
  moshi: 具体的な市町村間を結んでいるのかまでは、今のところ分からないんだ。
  人間: じゃあ、だいたいどのあたりを通ってるのかくらいは分かる？


会話の記録（`.txt`）を見る。

In [38]:
print((S02 / f"{STEM}.txt").read_text(encoding="utf-8"))

(unaugmented)
moshi: やあ。
Human: あの道の、一番長い区間ってどれなんだろう
(unaugmented)
moshi: 申し訳ないけど、川平有料道路の一番長い区間については知らないんだ。
Human: 他に何か分かってる区間とかないの？
(augmented)
moshi (lead): うーん、他の区間についてですね。
Reference: 川平有料道路は、長崎県北部の地域間を結び、地元の物流や観光客の移動を円滑化する役割を担っている。
moshi (body): 北部の地域間を結んでるってことは言えるけどね。
moshi (tail): [empty]
Human: 具体的にどのエリア間を走ってるのか教えてくれる？
(augmented)
moshi (lead): なるほど、エリア間のことですね。
Reference: 川平有料道路の路線情報は限定的で、具体的にどの市町村間を結ぶのかは判明していない。
moshi (body): 具体的な市町村間を結んでいるのかまでは、今のところ分からないんだ。
moshi (tail): [empty]
Human: じゃあ、だいたいどのあたりを通ってるのかくらいは分かる？



---
## s03　台本と構造 JSON に分ける

会話の記録を、TTS の台本・ターンの並び・lead と body の位置（struct.json）に分ける。

In [39]:
S03 = WORK / "s03"
s03.main(Namespace(input_dir=str(S02), output_dir=str(S03), limit=0))
print("\n--- 台本（.txt）---")
print((S03 / f"{STEM}.txt").read_text(encoding="utf-8"))

1 会話を変換 -> /home/jsato/moshirag-jp/data/_walkthrough/s03
  augmented ターン合計 2（1 会話あたり 2.0）

--- 台本（.txt）---
[S2]やあ。
[S1]あの道の、一番長い区間ってどれなんだろう
[S2]申し訳ないけど、川平有料道路の一番長い区間については知らないんだ。
[S1]他に何か分かってる区間とかないの？
[S2]うーん、他の区間についてですね。
[S2]北部の地域間を結んでるってことは言えるけどね。
[S1]具体的にどのエリア間を走ってるのか教えてくれる？
[S2]なるほど、エリア間のことですね。
[S2]具体的な市町村間を結んでいるのかまでは、今のところ分からないんだ。
[S1]じゃあ、だいたいどのあたりを通ってるのかくらいは分かる？



In [40]:
turns = json.loads((S03 / f"{STEM}.json").read_text(encoding="utf-8"))
struct = json.loads((S03 / f"{STEM}.struct.json").read_text(encoding="utf-8"))
lead_idx = {a["lead_turn_index"] for a in struct["augmented"]}
body_idx = {a["body_turn_index"] for a in struct["augmented"]}
pd.DataFrame([{"ターン": i, "話者": t["speaker"],
               "部分": "lead" if i in lead_idx else "body" if i in body_idx else "",
               "テキスト": t["text"]} for i, t in enumerate(turns)])

,ターン,話者,部分,テキスト
0,0,B,,やあ。
1,1,A,,あの道の、一番長い区間ってどれなんだろう
2,2,B,,申し訳ないけど、川平有料道路の一番長い区間については知らないんだ。
3,3,A,,他に何か分かってる区間とかないの？
4,4,B,lead,うーん、他の区間についてですね。
5,5,B,body,北部の地域間を結んでるってことは言えるけどね。
6,6,A,,具体的にどのエリア間を走ってるのか教えてくれる？
7,7,B,lead,なるほど、エリア間のことですね。
8,8,B,body,具体的な市町村間を結んでいるのかまでは、今のところ分からないんだ。
9,9,A,,じゃあ、だいたいどのあたりを通ってるのかくらいは分かる？


In [41]:
# 構造 JSON の augmented（s06 が使う）
pd.DataFrame([{k: (v[:45] + "…" if isinstance(v, str) and len(v) > 45 else v) for k, v in a.items()}
              for a in struct["augmented"]])

,lead_turn_index,body_turn_index,tail_turn_index,lead,body,tail,reference
0,4,5,None,うーん、他の区間についてですね。,北部の地域間を結んでるってことは言えるけどね。,,川平有料道路は、長崎県北部の地域間を結び、地元の物流や観光客の移動を円滑化する役割を担って…
1,7,8,None,なるほど、エリア間のことですね。,具体的な市町村間を結んでいるのかまでは、今のところ分からないんだ。,,川平有料道路の路線情報は限定的で、具体的にどの市町村間を結ぶのかは判明していない。


---
## s04　音声にする

s03 の台本を、TTS 専用 venv の `synth_worker.py`（FireRedTTS）で実際に音声化する。
会話のステレオ wav・manifest（ターンごとの開始秒）・ターンごとの wav ができる。

In [ ]:
AUDIO = WORK / "s04_audio"
# 台本（s03 の .txt）を TTS 専用 venv で音声化する。モデルは HF キャッシュから読む（HF_HUB_OFFLINE=1）。
# synth_worker は in_dir の *.txt を 1 件ずつ合成し、会話の wav・manifest.json・_turns/ を out_dir に書く。
run_step(
    [TTS_PY, TTS_SYNTH, "--in_dir", S03, "--out_dir", AUDIO],
    env={"TTS_DIR": str(TTS_PKG), "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
         "CUDA_VISIBLE_DEVICES": GPU},
    cwd=TTS_PKG, title="s04 音声合成",
)
manifest = json.loads((AUDIO / f"{STEM}.manifest.json").read_text(encoding="utf-8"))
pd.DataFrame(manifest["turns"])

---
## s05　語アライメント

MFA に渡すコーパス（ターンの wav と書き起こし）を作る。

In [ ]:
S05_WORK, WORDS = WORK / "s05_work", WORK / "s05_words"
bases = s05.build_corpus(AUDIO, [STEM], S05_WORK / "corpus")
print(f"コーパス {len(bases)} ターン")
for p in sorted((S05_WORK / "corpus").rglob("*.lab"))[:5]:
    print(f"  {p.relative_to(S05_WORK)}  →  {p.read_text(encoding='utf-8').strip()}")

コーパスを実際の MFA（日本語 acoustic・辞書）で揃える。専用 conda 環境の `mfa align` を呼び、
ターンごとの語 JSON（`out/A|B/*.json`）を作る。次のセルでこれを会話全体の時間軸にまとめる。

In [ ]:
# s05.build_corpus が作ったコーパスを、実際の MFA で揃える（広いビーム beam 100 / retry 400 の 1 回）。
# MFA_ROOT_DIR に日本語モデルと設定を置き、conda 環境の mfa を PATH の先頭にして呼ぶ。
(S05_WORK / "out").mkdir(parents=True, exist_ok=True)
run_step(
    [MFA_ENV / "bin" / "mfa", "align", "--clean", "-j", "1", "--output_format", "json",
     "-c", BEAM_YAML, S05_WORK / "corpus", "japanese_mfa", "japanese_mfa", S05_WORK / "out"],
    env={"MFA_ROOT_DIR": str(MFA_ROOT), "PATH": f"{MFA_ENV}/bin:{os.environ['PATH']}"},
    title="s05 MFA アライン",
)

sample = next((S05_WORK / "out").glob("*/*.json"))
print(sample.relative_to(S05_WORK), ":")
print(json.dumps(json.loads(sample.read_text(encoding="utf-8")), ensure_ascii=False)[:300], "…")

In [ ]:
st05 = s05.to_words(AUDIO, [STEM], [S05_WORK / "out"], WORDS)
print(dict(st05))
words = json.loads((WORDS / f"{STEM}.json").read_text(encoding="utf-8"))
pd.DataFrame(words).head(16)

---
## s06　`<ret>` の位置と lead の長さを求める

manifest と構造 JSON から `ret_frame` と `d_lead` を求める。

In [ ]:
RET = WORK / "s06_ret"
s06.main(Namespace(manifest_dir=str(AUDIO), struct_dir=str(S03), output_dir=str(RET)))
ret = json.loads((RET / f"{STEM}.json").read_text(encoding="utf-8"))
pd.DataFrame(ret["ret_events"])[["lead_turn_index", "body_turn_index", "lead_onset", "body_onset",
                                  "d_lead", "ret_frame", "d_lead_frames", "lead"]]

時間軸に並べて見る。

In [ ]:
def plot_timeline(manifest, struct, events, title=""):
    lead_i = {a["lead_turn_index"] for a in struct["augmented"]}
    body_i = {a["body_turn_index"] for a in struct["augmented"]}
    fig, ax = plt.subplots(figsize=(13, 2.8))
    for t in manifest["turns"]:
        y = 1 if t["channel"] == 0 else 0
        c = ("#9aa0a6" if y == 1 else "#f29900" if t["index"] in lead_i
             else "#1e8e3e" if t["index"] in body_i else "#4285f4")
        ax.barh(y, t["duration"], left=t["onset"], color=c, edgecolor="white", height=0.6)
        ax.text(t["onset"] + 0.05, y, str(t["index"]), va="center", fontsize=8, color="white")
    for e in events:
        x = e["ret_frame"] / FRAME_RATE
        ax.axvline(x, color="crimson", ls="--", lw=1)
        ax.annotate("", xy=(e["body_onset"], -0.45), xytext=(e["lead_onset"], -0.45),
                    arrowprops=dict(arrowstyle="<->", color="#f29900"))
        ax.text((e["lead_onset"] + e["body_onset"]) / 2, -0.62, f"d_lead {e['d_lead']:.2f}s",
                ha="center", fontsize=8, color="#b06000")
    ax.set_yticks([0, 1], ["B：Moshi 役（右）", "A：利用者役（左）"])
    ax.set_ylim(-0.8, 1.5)
    ax.set_xlabel("会話の先頭からの秒数（数字はターン番号）")
    ax.legend(handles=[Patch(color="#9aa0a6", label="利用者役"), Patch(color="#4285f4", label="Moshi（unaugmented）"),
                       Patch(color="#f29900", label="lead"), Patch(color="#1e8e3e", label="body"),
                       Line2D([0], [0], color="crimson", ls="--", label="ret_frame（s06）")],
              loc="upper center", bbox_to_anchor=(0.5, 1.28), ncol=5, fontsize=8, frameon=False)
    ax.set_title(title, fontsize=10, pad=28)
    plt.tight_layout()
    plt.show()


plot_timeline(manifest, struct, ret["ret_events"], "s04 の manifest（ダミー）と s06 の結果")

---
## s07　音声トークンにする

会話のステレオ wav を Mimi で離散トークンにする（A・B それぞれ `[8, T]`、12.5 Hz）。
torch+kame の venv で GPU を使う。

In [ ]:
TOK_AUDIO = WORK / "s07_tok_audio"
TOK_AUDIO.mkdir()
# 会話のステレオ wav を Mimi でトークナイズする（torch+kame の venv、GPU）。
# tokenize_audio は audio_dir 直下の *.wav を読む（_turns/ の中は見ない）ので、会話の wav だけが対象になる。
run_step(
    [TOK_PY, "-m", "moshirag_data.s07_tok_audio",
     "--audio_dir", AUDIO, "--output_dir", TOK_AUDIO, "--num_workers", "1"],
    env={"CUDA_VISIBLE_DEVICES": GPU}, cwd=REPO, title="s07 音声トークン",
)
au = np.load(TOK_AUDIO / f"{STEM}.npz")
T_audio = au["A"].shape[1]
print(f"A {au['A'].shape} / B {au['B'].shape} → {T_audio} フレーム")

---
## s08　テキストトークンにする

語の時刻から、12.5 Hz のテキストの並び（PAD・EPAD 入り）を作る。

In [ ]:
TOK_TEXT = WORK / "s08_tok_text"
TOK_TEXT.mkdir()
args08 = Namespace(word_transcript_dir=str(WORDS), output_dir=str(TOK_TEXT),
                   text_tokenizer_repo="llm-jp/j-moshi-v1", text_tokenizer_name="tokenizer_spm_32k_3.model",
                   no_whitespace_before_word=True, text_padding_id=3, end_of_text_padding_id=0,
                   audio_tokenizer_frame_rate=FRAME_RATE, num_workers=1, resume=False,
                   allow_missing_speakers=False, allow_alignment_warnings=True)
tok_text.worker(0, [STEM], args08)

tt = np.load(TOK_TEXT / f"{STEM}.npz")
for spk in ("A", "B"):
    x = tt[spk]
    print(f"{spk}: {len(x)} フレーム / PAD {np.sum(x == 3)} / EPAD {np.sum(x == 0)} / テキスト {np.sum((x != 3) & (x != 0))}")
print(f"（音声は {T_audio} フレーム）")

In [ ]:
from huggingface_hub import hf_hub_download
from sentencepiece import SentencePieceProcessor

sp = SentencePieceProcessor(hf_hub_download(args08.text_tokenizer_repo, args08.text_tokenizer_name))
SPECIAL = {3: "PAD", 0: "EPAD", 4: "<ret>"}


def piece(i):
    return SPECIAL.get(int(i), sp.id_to_piece(int(i)))


def stream_view(x, start, n=22, mark=None):
    # テキストの並びの一部を横に並べて見る
    fr = list(range(max(start, 0), min(start + n, len(x))))
    df = pd.DataFrame({"秒": [f"{f / FRAME_RATE:.2f}" for f in fr], "ID": [int(x[f]) for f in fr],
                       "トークン": [piece(x[f]) for f in fr]}, index=fr).T
    df.columns.name = "フレーム"
    if mark is not None:
        return df.style.format(escape="html").apply(lambda c: ["background-color:#fde293" if c.name in mark else "" for _ in c])
    return df


e0 = ret["ret_events"][0]
print(f"B（Moshi 役）の 1 つ目の lead「{e0['lead']}」のあたり。s06 の ret_frame = {e0['ret_frame']}")
stream_view(tt["B"], e0["ret_frame"] - 3, mark={e0["ret_frame"]})

---
## s09　参照をベクトルにする

`<ret>` ごとの参照文を ARC-Encoder で符号化する（参照ごとに `[n/4, 3072]`、bridge の手前の凍結部分）。
torch の venv で GPU を使う。

In [ ]:
REF = WORK / "s09_ref_embed"
REF.mkdir()
stems_file = WORK / "stems.txt"
stems_file.write_text(STEM + "\n", encoding="utf-8")
# s06 の ret_events の参照文を ARC-Encoder で符号化する（torch の venv、GPU）。
# 初回は ARC のソース（小さい .py）を GitHub から取り、重みは HF キャッシュから読む。
run_step(
    [TOK_PY, REPO / "moshirag_data" / "s09_ref_embed.py",
     "--ret_dir", RET, "--stems", stems_file, "--out", REF, "--cache", WORK / "_arc_cache"],
    env={"CUDA_VISIBLE_DEVICES": GPU}, cwd=REPO, title="s09 参照ベクトル",
)
rf = np.load(REF / f"{STEM}.npz")
print({k: (rf[k].shape, rf[k].dtype.name) for k in rf.files}, "/ lengths =", rf["lengths"].tolist())

---
## s10　parquet にまとめる

s06〜s09 を 1 行にまとめ、lead の直前の EPAD を `<ret>` に置き換える。

In [ ]:
stems_file = WORK / "stems.txt"
stems_file.write_text(STEM + "\n", encoding="utf-8")
S10 = WORK / "s10_parquet"
argv = ["s10_pack.py", "--tokenized_text_dir", str(TOK_TEXT), "--tokenized_audio_dir", str(TOK_AUDIO),
        "--ref_embed_dir", str(REF), "--ret_dir", str(RET), "--stems", str(stems_file),
        "--output_prefix", str(S10 / "moshirag_jp_walkthrough")]
with mock.patch.object(sys, "argv", argv):
    s10.main()

parquet を読み戻して中身を見る。

In [ ]:
pq = sorted(S10.glob("*.parquet"))[0]
row = pd.read_parquet(pq).iloc[0]
codes = np.frombuffer(row["codes"], np.int32).reshape(tuple(row["codes_shape"]))
ret_frames = np.frombuffer(row["ret_frames"], np.int32)
d_lead_frames = np.frombuffer(row["d_lead_frames"], np.int32)
ref_offsets = np.frombuffer(row["ref_offsets"], np.int32)
ref_values = np.frombuffer(row["ref_values"], np.float16).reshape(-1, row["ref_dim"])

print(pq.name)
pd.DataFrame([
    {"列": "dialogue_id", "形": "", "中身": row["dialogue_id"]},
    {"列": "codes", "形": str(codes.shape), "中身": "行 0 テキスト / 行 1〜8 Moshi 役の音声 / 行 9〜16 利用者役の音声"},
    {"列": "ret_frames", "形": str(ret_frames.shape), "中身": ret_frames.tolist()},
    {"列": "d_lead_frames", "形": str(d_lead_frames.shape), "中身": d_lead_frames.tolist()},
    {"列": "ref_offsets", "形": str(ref_offsets.shape), "中身": ref_offsets.tolist()},
    {"列": "ref_values", "形": str(ref_values.shape), "中身": f"float16, ref_dim = {row['ref_dim']}"},
])

In [ ]:
# 行 1〜8 が s07 の B、行 9〜16 が A と一致すること、行 0 に <ret> が置かれたことを確かめる
assert (codes[1:9] == au["B"][:, :codes.shape[1]]).all() and (codes[9:17] == au["A"][:, :codes.shape[1]]).all()
pd.DataFrame([{"lead": e["lead"], "s06 の ret_frame": e["ret_frame"], "s10 で置いた位置": int(p),
               "置く前のトークン": piece(tt["B"][p]), "置いた後": piece(codes[0, p]),
               "次のトークン（lead の先頭）": piece(codes[0, p + 1])}
              for e, p in zip(ret["ret_events"], ret_frames)])

In [ ]:
p0 = int(ret_frames[0])
print("parquet の行 0（テキスト）の、1 つ目の <ret> のあたり")
stream_view(codes[0], p0 - 3, mark={p0})

---
## 参考：学習時に参照を足し始める位置

`<ret>` からの遅れ d′ の分布を見る（`moshirag/moshirag_bridge.py` の式を写したもの）。

In [ ]:
def sample_delay_frames(d_lead_frames: int, rng: np.random.Generator) -> int:
    # moshirag/moshirag_bridge.py の同名の関数と同じ式（論文 式 3）
    d_lead = d_lead_frames / FRAME_RATE
    if d_lead < 2.0 or rng.random() < 0.2:
        d = rng.uniform(0.0, d_lead)
    else:
        d = rng.uniform(1.0, d_lead - 1.0)
    return int(round(d * FRAME_RATE))


g = np.random.default_rng(0)
fig, axes = plt.subplots(1, len(ret_frames), figsize=(5 * len(ret_frames), 2.4), squeeze=False)
for ax, rf_k, dl in zip(axes[0], ret_frames, d_lead_frames):
    starts = [rf_k + sample_delay_frames(int(dl), g) for _ in range(5000)]
    ax.hist(starts, bins=range(rf_k, rf_k + int(dl) + 2), color="#4285f4")
    ax.axvline(rf_k, color="crimson", ls="--", lw=1, label="<ret>")
    ax.axvline(rf_k + dl, color="#1e8e3e", lw=1.5, label="body の始まり（≒ <ret> + d_lead）")
    ax.set_title(f"<ret> = フレーム {rf_k}、d_lead = {dl / FRAME_RATE:.2f} 秒", fontsize=9)
    ax.set_xlabel("参照を足し始めるフレーム（5,000 回引いた分布）")
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

---
## 作ったファイル

In [ ]:
ls(WORK)